# 02 — Data Readiness

**Owner:** Person A (modelling track).

**Rubric line:** Data quality, target definition, train/test split, data
dictionary, reproducibility.

Loads the raw CSV, applies safe pre-split cleaning, splits 80/20
stratified, and persists to `data/interim/` so downstream notebooks load
the same train/test pair.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 2.1 — Load raw data

In [ ]:
df_raw = data.load_raw()
print(df_raw.shape)
df_raw.head()


## 2.2 — Profile: shape, dtypes, missing values, target balance

In [ ]:
print(df_raw.info())
print('\nTarget distribution:')
print(df_raw[config.TARGET_COL].value_counts(normalize=True))


## 2.3 — Light cleaning (safe to do BEFORE the split)

- Encode target as 0/1.
- Replace `pdays == 999` sentinel with NaN, add `was_previously_contacted` flag.
- Strip whitespace from string cells.

**Note:** any imputation, scaling, or encoding that *learns from data* is
applied AFTER the split inside the modelling pipeline (see `src/features.py`).
We never fit on the full dataset.


In [ ]:
df_clean = data.light_clean(df_raw)
df_clean.head()


## 2.4 — Train/test split (stratified 80/20)

In [ ]:
train_df, test_df = data.split_train_test(df_clean)
print(f'Train: {train_df.shape}  positive rate: {train_df[config.TARGET_COL].mean():.3%}')
print(f'Test:  {test_df.shape}  positive rate: {test_df[config.TARGET_COL].mean():.3%}')


## 2.5 — Persist to data/interim/

In [ ]:
data.save_interim(train_df, test_df)


## 2.6 — Data dictionary (1–2 lines per field)

| Column | Type | Description |
|--------|------|-------------|
| age | numeric | Customer age in years. |
| job | categorical | Occupation category (12 levels including 'unknown'). |
| marital | categorical | Marital status (married/single/divorced/unknown). |
| education | categorical | Highest education level. |
| default | categorical | Has credit in default? (yes/no/unknown). |
| housing | categorical | Has housing loan? (yes/no/unknown). |
| loan | categorical | Has personal loan? (yes/no/unknown). |
| contact | categorical | Communication channel (cellular/telephone). |
| month | categorical | Month of last contact (jan–dec). |
| day_of_week | categorical | Day of last contact (mon–fri). |
| duration | numeric | **LEAKY.** Last call duration in seconds — known only after the call. Excluded from the deployable model. |
| campaign | numeric | Number of contacts during this campaign for this customer. |
| pdays | numeric | Days since last contact in a previous campaign (NaN = never contacted). |
| previous | numeric | Number of contacts in previous campaigns. |
| poutcome | categorical | Outcome of the previous campaign (success/failure/nonexistent). |
| emp.var.rate | numeric | Employment variation rate (quarterly macro indicator). |
| cons.price.idx | numeric | Consumer price index (monthly macro). |
| cons.conf.idx | numeric | Consumer confidence index (monthly macro). |
| euribor3m | numeric | 3-month Euribor rate (daily macro). |
| nr.employed | numeric | Number of employees, in thousands (quarterly macro). |
| was_previously_contacted | derived | 1 if customer was contacted in a prior campaign, else 0. |
| **y** | **target** | **1 if the customer subscribed to the term deposit, else 0.** |
